

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akshayrb22/playing-with-data/blob/master/supervised_learning/KNN/KNN.ipynb)

# K-Nearest Neighbors

Baseado em: https://colab.research.google.com/github/akshayrb22/playing-with-data/blob/master/supervised_learning/KNN/KNN.ipynb

  O algoritmo dos K-vizinhos mais próximos (KNN) é um tipo de algoritmo de aprendizado de máquina supervisionado.
  
  O KNN é extremamente fácil de implementar em sua forma mais básica e, ainda assim, realiza tarefas de classificação bastante complexas.
  
  Trata-se de um algoritmo de aprendizado preguiçoso (lazy learning), pois não possui uma fase de treinamento especializada. Em vez disso, utiliza todos os dados disponíveis para classificar um novo ponto ou instância. O KNN é um algoritmo de aprendizado não paramétrico, o que significa que não assume nenhuma hipótese sobre a distribuição dos dados subjacentes. Essa é uma característica extremamente útil, já que a maioria dos dados do mundo real não segue, necessariamente, pressupostos teóricos, como separabilidade linear, distribuição uniforme, etc.

## ✅ Vantagens

- É extremamente fácil de implementar.
- Como mencionado anteriormente, trata-se de um algoritmo de aprendizado preguiçoso (lazy learning) e, portanto, não requer treinamento prévio para fazer previsões em tempo real. Isso torna o KNN muito mais rápido do que outros algoritmos que exigem treinamento, como SVM, regressão linear, etc.
- Como o algoritmo não exige treinamento, novos dados podem ser adicionados de forma contínua e sem complicações.
- São necessários apenas dois parâmetros para implementar o KNN: o valor de K e a função de distância (por exemplo, Euclidiana, Manhattan, etc.).

## ❌ Desvantagens

- O KNN não funciona bem com dados de alta dimensionalidade, pois, com muitas dimensões, torna-se difícil calcular distâncias de forma eficaz.
- O algoritmo KNN possui alto custo de predição em grandes conjuntos de dados, já que precisa calcular a distância entre o novo ponto e todos os pontos existentes.
- Por fim, o KNN não lida bem com variáveis categóricas, pois é difícil calcular distâncias entre atributos categóricos.

In [ ]:
#Importing necessary libraries
import pandas as pd
import numpy as np
import math
import operator

# Importing the Dataset

## 📊 Dataset: Breast Cancer Wisconsin (Diagnostic)

O conjunto de dados Breast Cancer Wisconsin (Diagnostic) é amplamente utilizado para treinar e avaliar algoritmos de classificação supervisionada. Ele contém informações extraídas de imagens digitalizadas de biópsias de massas mamárias, com o objetivo de prever se uma amostra é maligna (cancerosa) ou benigna (não cancerosa).

## ✅ Características principais:

- Número de instâncias: 569
- Número de atributos: 30 características numéricas extraídas das imagens (como textura, área, suavidade, compacidade, simetria, etc.)
- Atributo alvo (target):
 - M = Maligno
 - B = Benigno
- Tipo de problema: Classificação binária (supervisionado)
- Fonte: UCI Machine Learning Repository

## 🎯 Objetivo:

Utilizar as medidas extraídas das imagens das células para prever corretamente se o tumor é maligno ou benigno, auxiliando no diagnóstico médico precoce.

In [ ]:
url = 'https://raw.githubusercontent.com/melwinlobo18/K-Nearest-Neighbors/master/Dataset/data.csv'
df = pd.read_csv(url)  # Dataset - Breast Cancer Wisconsin Data
df['diagnosis'] = df['diagnosis'].map({
    'M': 1,
    'B': 2
})  # Label values - 1 for Malignant and 2 for Benign
labels = df['diagnosis'].tolist()
df['Class'] = labels  #Cpying values of diagnosis to newly clreated labels column
df = df.drop(['id', 'Unnamed: 32', 'diagnosis'],
             axis=1)  #Dropping unncessary columns
df.head()  #Displaying first five rows of the dataset

,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Class
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,1
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,1
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,1
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,1
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,1


# Train-Test Split

In [ ]:
np.random.seed(1)
msk = np.random.rand(
    len(df)) < 0.7  #An array containing True(with probability 0.7) and False
train = df[msk]  #Rows having array value true
test = df[~msk]  #Rows having array value False
print('Number of observations in the training data: ', len(train))
print('Number of observations in the test data: ', len(test))

Number of observations in the training data:  395
Number of observations in the test data:  174


In [ ]:
train.head()

,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Class
0,17.99,10.38,122.80,1001.0,0.1184,0.2776,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,1
2,19.69,21.25,130.00,1203.0,0.1096,0.1599,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,1
3,11.42,20.38,77.58,386.1,0.1425,0.2839,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,1
4,20.29,14.34,135.10,1297.0,0.1003,0.1328,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,1
5,12.45,15.70,82.57,477.1,0.1278,0.1700,0.1578,0.08089,0.2087,0.07613,...,23.75,103.40,741.6,0.1791,0.5249,0.5355,0.1741,0.3985,0.12440,1


In [ ]:
test.head()

,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Class
1,20.570,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.24160,0.18600,0.2750,0.08902,1
13,15.850,23.95,103.70,782.7,0.08401,0.10020,0.09938,0.05364,0.1847,0.05338,...,27.66,112.00,876.5,0.1131,0.1924,0.23220,0.11190,0.2809,0.06287,1
20,13.080,15.71,85.63,520.0,0.10750,0.12700,0.04568,0.03110,0.1967,0.06811,...,20.49,96.09,630.5,0.1312,0.2776,0.18900,0.07283,0.3184,0.08183,2
21,9.504,12.44,60.34,273.9,0.10240,0.06492,0.02956,0.02076,0.1815,0.06905,...,15.66,65.13,314.9,0.1324,0.1148,0.08867,0.06227,0.2450,0.07773,2
24,16.650,21.38,110.00,904.6,0.11210,0.14570,0.15250,0.09170,0.1995,0.06330,...,31.56,177.00,2215.0,0.1805,0.3578,0.46950,0.20950,0.3613,0.09564,1


# Euclidean Distance

É usada para medir a distância entre as amostras p e q em um espaço de características com n dimensões:

<img src="https://drive.google.com/uc?export=view&id=13p3YOEqHybBrN8P5wpsPwBC6-7v3N98f">

Por exemplo, imagine-a como uma linha "reta e conectando" os pontos em um espaço de características bidimensional (2D).

<img src="https://drive.google.com/uc?export=view&id=1Qc0nBRJTWBrJeKZvvj8dVk7YZJrvMpn8">

Implemente uma função chamada distancia_euclidiana(p, q)
p e q são listas ou vetores com n valores numéricos.
A função deve retornar a distância euclidiana entre p e q.


💡 Dica:
Você pode utilizar a função math.sqrt() e um laço for ou uma função com zip() para iterar pelos elementos dos vetores.

In [ ]:
def euclideanDistance(instance1, instance2):
    distance = 0
    for p, q in zip(instance1, instance2):
      distance = distance + (p-q)**2

    distance = math.sqrt(distance)
    return distance

In [ ]:
# Teste 1
p1 = [0, 0]
q1 = [3, 4]
print(euclideanDistance(p1, q1))

# Teste 2
p2 = [1, 2, 3]
q2 = [4, 0, 3]
print(euclideanDistance(p2, q2))

# Teste 3
p3 = [7]
q3 = [2]
print(euclideanDistance(p3, q3))

5.0
3.605551275463989
5.0


Resposta esperada:

```
5.0
3.605551275463989
5.0
```

In [ ]:
p1 = [0, 0]
q1 = [3, 4]

z = zip(p1, q1)

print(p1)
print(q1)
print(list(z))

[0, 0]
[3, 4]
[(0, 3), (0, 4)]


# Calculando os vizinhos mais próximos (Nearest Neighbors)

Vamos considerar que nossos dados estão distribuídos conforme mostrado abaixo. Os círculos vermelhos representam casos malignos e os círculos verdes representam casos benignos:

<img src="https://drive.google.com/uc?export=view&id=1oEbIjxE0Ysdo5fEeNU5CGo4ruP544Ywz">

Neste exemplo, estamos calculando com k = 3 vizinhos mais próximos, então a curva de ajuste ficará assim:

<img src="https://drive.google.com/uc?export=view&id=1amCw22CqF0ideK9VWNIWQAASpZOWGPwi">

A função ```getNeighbors``` retorna os k vizinhos mais próximos de uma instância de teste, com base na distância euclidiana entre essa instância e os elementos de um conjunto de treinamento.

### Parâmetros:

- trainingSet: lista de instâncias (listas ou vetores) já rotuladas.
- testInstance: instância para a qual queremos prever ou analisar os vizinhos mais próximos.
- k: número de vizinhos mais próximos que devem ser retornados.

### Funcionamento:

- Calcula a distância euclidiana entre a testInstance e cada ponto do trainingSet.
- Armazena essas distâncias em uma lista de tuplas (instância, distância).
- Ordena a lista com base na menor distância.
Retorna as k instâncias mais próximas.

### Importante: A função assume que o último elemento de cada vetor representa o rótulo (classe), você deve remover essa informação para calcular a distância.

In [ ]:
def getNeighbors(trainingSet, testInstance, k):
    distances = []  #List to store all the distance values
    neighbors = []  #List to store all the neighbors
    #calcular a distância para todos os elementos do conjunto de treinamento
    for i, instance in enumerate(trainingSet):
      dist = euclideanDistance(instance[:-1], testInstance[:-1])
      distances.append([i, dist])

    distances.sort(key=lambda x: x[1])

    neighbors = []
    for i in range(k):
      neighbors.append(trainingSet[distances[i][0]])

    return neighbors

In [ ]:
distances = [[0, 1.2],[1, 2.5],[2, 0.6],[3, 0.2]]
print("Antes do ordenamento:", distances)
distances.sort(key=lambda x: x[1])
print("Depois do ordenamento:", distances)

Antes do ordenamento: [[0, 1.2], [1, 2.5], [2, 0.6], [3, 0.2]]
Depois do ordenamento: [[3, 0.2], [2, 0.6], [0, 1.2], [1, 2.5]]


In [ ]:
#Código para testar sua solução
# Conjunto de treinamento: [feature1, feature2, classe]
trainingSet = [
    [2, 4, 'A'],
    [4, 4, 'B'],
    [4, 6, 'B'],
    [6, 6, 'A']
]

# Instância de teste
testInstance = [5, 5, None]

# Obter os 3 vizinhos mais próximos
neighbors = getNeighbors(trainingSet, testInstance, k=3)

# Exibir os vizinhos
for neighbor in neighbors:
    print(neighbor)

[4, 4, 'B']
[4, 6, 'B']
[6, 6, 'A']


Resposta esperada:
```
[4, 4, 'B']
[4, 6, 'B']
[6, 6, 'A']
```

## Códigos para teste

In [ ]:
classVotes = {}  #Dictionary to store labels with their counts
for n in neighbors:
  label_n = n[-1]
  if label_n in classVotes:
    classVotes[label_n] += 1
  else:
    classVotes[label_n] = 1

classVotes

{'B': 2, 'A': 1}

In [ ]:
classVotes = {'B': 2, 'A': 1, 'C':100}

In [ ]:
keys = list(classVotes.keys())
classe_vencedora = keys[0]

for k in keys[1:]:
  if classVotes[k] > classVotes[classe_vencedora]:
    classe_vencedora = k

classe_vencedora

'C'

A função getResponse é usada para decidir a classe final de uma nova instância com base nos rótulos dos k vizinhos mais próximos retornados pela função getNeighbors.

### ✏️ Objetivo da tarefa:

Entender como, a partir dos vizinhos mais próximos, o algoritmo KNN "vota" na classe mais frequente.

### 🧩 Descrição da função getResponse(neighbors)
- A função recebe como entrada uma lista de neighbors, onde cada elemento é uma instância do conjunto de treinamento com seu respectivo rótulo (último elemento da lista).
- Ela contabiliza o número de ocorrências de cada classe entre os vizinhos.
- A classe com maior número de votos é retornada como a previsão final para a instância de teste.

In [ ]:
def getResponse(neighbors):
    classVotes = {}  #Dictionary to store labels with their counts
    for n in neighbors:
      label_n = n[-1]
      if label_n in classVotes:
        classVotes[label_n] += 1
      else:
        classVotes[label_n] = 1
    #TODO: preencher o dicionário com a quantidade de vizinhos de cada classe
    sortedVotes = sorted(
        classVotes.items(), key=operator.itemgetter(1), reverse=True
    )  #Sort the dictinary based on the count value in descending order
    return sortedVotes[0][0]  #Return the label with highest number of occurences

In [ ]:
# Teste 1: Classe majoritária é 'A'
neighbors1 = [
    [2, 4, 'A'],
    [4, 4, 'B'],
    [4, 6, 'A']
]
print(getResponse(neighbors1))  # Esperado: 'A'

# Teste 2: Classe majoritária é 'B'
neighbors2 = [
    [1, 2, 'B'],
    [2, 3, 'B'],
    [3, 1, 'A']
]
print(getResponse(neighbors2))  # Esperado: 'B'

# Teste 3: Empate entre 'A' e 'B' (o retorno dependerá da ordem no dicionário)
neighbors3 = [
    [1, 1, 'A'],
    [2, 2, 'B']
]
print(getResponse(neighbors3))  # Pode retornar 'A' ou 'B' (ordem importa)

A
B
A


Resposta esperada
```
A
B
A
```

# Calculando a acurácia

A acurácia é uma métrica utilizada para avaliar modelos de aprendizado de máquina. A acurácia representa a fração de previsões que o nosso modelo acertou. Formalmente, a acurácia é definida da seguinte maneira:

<img src="https://drive.google.com/uc?export=view&id=1gk-T6kanOB6mM_bs54jgyMs0Bu93V4sh">

In [ ]:
def getAccuracy(testSet, predictions):
    correct = 0  #Variable to store the correct predictions
    for x in range(len(testSet)):
        if testSet[x][-1] is predictions[x]:  #Checking whether the predicted value is same as label value
            correct += 1  #Incremented when both values are same
    return (correct / float(len(testSet))) * 100.0  #Accuracy = No. of Correct pred / Total number of pred

# Testando em um dataset pequeno




In [ ]:
trainSet = [[5, 1, 1, 1, 2, 1, 3, 2, 1, 2],
            [10, 10, 10, 10, 5, 10, 10, 10, 7, 4]]
testInstance = [4, 8, 6, 4, 3, 4, 10, 6, 1, 2]
k = 1
neighbors = getNeighbors(trainSet, testInstance, k)
print(neighbors)

[[5, 1, 1, 1, 2, 1, 3, 2, 1, 2]]


In [ ]:
neighbors = [[5, 1, 1, 1, 2, 1, 3, 2, 1, 2], [3, 1, 1, 1, 2, 1, 2, 3, 1, 2],
             [10, 10, 10, 10, 5, 10, 10, 10, 7, 4]]
response = getResponse(neighbors)
print(response)

2


In [ ]:
testSet = [[5, 1, 1, 1, 2, 1, 3, 2, 1, 2], [3, 1, 1, 1, 2, 1, 2, 3, 1, 2],
           [10, 10, 10, 10, 5, 10, 10, 10, 7, 4]]
predictions = [2, 2, 2]
accuracy = getAccuracy(testSet, predictions)
print(accuracy)

66.66666666666666


# Executando no dataset alvo

Execute o código abaixo e varie o parâmetro k. O que você consegue oberservar?

In [ ]:
predictions = []  #List to store the predicted values
k = 3  # 3-Nearest Neighbors
trainingSet = train.values.tolist()  #List containing training data
testSet = test.values.tolist()  #List containing test data

In [ ]:
testSet[:2]

[[20.57,
  17.77,
  132.9,
  1326.0,
  0.08474,
  0.07864,
  0.0869,
  0.07017,
  0.1812,
  0.05667,
  0.5435,
  0.7339,
  3.398,
  74.08,
  0.005225,
  0.01308,
  0.0186,
  0.0134,
  0.01389,
  0.003532,
  24.99,
  23.41,
  158.8,
  1956.0,
  0.1238,
  0.1866,
  0.2416,
  0.186,
  0.275,
  0.08902,
  1.0],
 [15.85,
  23.95,
  103.7,
  782.7,
  0.08401,
  0.1002,
  0.09938,
  0.05364,
  0.1847,
  0.05338,
  0.4033,
  1.078,
  2.903,
  36.58,
  0.009769,
  0.03126,
  0.05051,
  0.01992,
  0.02981,
  0.003002,
  16.84,
  27.66,
  112.0,
  876.5,
  0.1131,
  0.1924,
  0.2322,
  0.1119,
  0.2809,
  0.06287,
  1.0]]

In [ ]:
predictions = []
for x in range(len(testSet)):
    neighbors = getNeighbors(trainingSet, testSet[x], k)
    result = getResponse(neighbors)
    predictions.append(result)  # Storing the predicted values
    print(f'> {x=} ' + 'predicted=' + repr(result) + ', actual=' + repr(testSet[x][-1]))

> x=0 predicted=1.0, actual=1.0
> x=1 predicted=2.0, actual=1.0
> x=2 predicted=2.0, actual=2.0
> x=3 predicted=2.0, actual=2.0
> x=4 predicted=1.0, actual=1.0
> x=5 predicted=1.0, actual=1.0
> x=6 predicted=1.0, actual=1.0
> x=7 predicted=1.0, actual=1.0
> x=8 predicted=2.0, actual=2.0
> x=9 predicted=2.0, actual=1.0
> x=10 predicted=1.0, actual=1.0
> x=11 predicted=2.0, actual=1.0
> x=12 predicted=1.0, actual=1.0
> x=13 predicted=2.0, actual=2.0
> x=14 predicted=2.0, actual=2.0
> x=15 predicted=1.0, actual=1.0
> x=16 predicted=1.0, actual=1.0
> x=17 predicted=2.0, actual=2.0
> x=18 predicted=1.0, actual=1.0
> x=19 predicted=2.0, actual=2.0
> x=20 predicted=2.0, actual=2.0
> x=21 predicted=1.0, actual=1.0
> x=22 predicted=1.0, actual=1.0
> x=23 predicted=1.0, actual=1.0
> x=24 predicted=2.0, actual=1.0
> x=25 predicted=2.0, actual=2.0
> x=26 predicted=2.0, actual=2.0
> x=27 predicted=2.0, actual=2.0
> x=28 predicted=2.0, actual=2.0
> x=29 predicted=2.0, actual=2.0
> x=30 predicted=2.0

# Matriz de Confusão

A Matriz de Confusão é uma métrica de desempenho para problemas de classificação em aprendizado de máquina, onde a saída pode pertencer a duas ou mais classes. Trata-se de uma tabela que apresenta quatro combinações possíveis entre os valores previstos e os valores reais.

<img src="https://drive.google.com/uc?export=view&id=1hYthqmYW82QHRFnIpiGQDLt4r44aurv7">

In [ ]:
print("Confusion Matrix")
y_test = []
for i in testSet:
    y_test.append(i[30])


Confusion Matrix


In [ ]:
len(predictions)

174

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score
res = confusion_matrix(y_test, predictions)
print(res)

[[ 52   5]
 [  6 111]]


No contexto deste dataset, qual tipo de erro é mais danoso para o usuário?

# Accuracy

In [ ]:
accuracy = accuracy_score(y_test, predictions) * 100
print('Accuracy: ' + repr(accuracy) + '%')

Accuracy: 93.67816091954023%
